# Génération de défauts synthétiques — IND-3

Baseline **copy-paste de masques réels + alpha blending** (méthode minimale).

- Images normales : `train/good/` (le split test ne sert jamais à entraîner un générateur).
- Défauts : `test/<défaut>/` + masques `ground_truth/<défaut>/*_mask.png` (référence de style uniquement).
- Sortie : `./synthetiques/<catégorie>/synth_XXX.png` (256×256), un `LOG.txt` et une grille d'exemples.

⚠️ Ce notebook est indépendant : il ne modifie pas `notebook_IND3_diffusion_defauts.ipynb`.

## 1. Imports + constantes

In [1]:
import random
from datetime import datetime
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image

# ===== Constantes (tout se règle ici) =====
SEED = 42
DATA_DIR = "./data"
SYNTH_DIR = "./synthetiques"
FIGURES_DIR = "./figures"
CATEGORIES = ["bottle", "cable", "transistor"]
N_SYNTH_PAR_CATEGORIE = 120          # entre 50 et 200, >= nombre de vrais défauts
TAILLE = 256                         # taille de sortie
MARGE_BBOX = 4                       # marge (px) autour du masque
FLOU_MASQUE = 9                      # noyau gaussien pour adoucir les bords
FORCE_DEFAUT = 0.95                  # opacité du défaut (0-1)
JITTER_MAX = 8                       # translation aléatoire (px)
VARIATION_LUMINOSITE = 0.03          # +/- gain de luminosité global
BRUIT_GAUSS = 1.5                    # bruit gaussien global
N_EXEMPLES_GRILLE = 5

random.seed(SEED)
np.random.seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

for d in [SYNTH_DIR, FIGURES_DIR]:
    Path(d).mkdir(parents=True, exist_ok=True)

print("Device :", DEVICE)
print("Catégories :", CATEGORIES)
print("Images synthétiques / catégorie :", N_SYNTH_PAR_CATEGORIE)
print("Taille de sortie :", TAILLE)

Device : cuda
Catégories : ['bottle', 'cable', 'transistor']
Images synthétiques / catégorie : 120
Taille de sortie : 256


## 2. Utilitaires : lecture images / masques

In [2]:
def lister_paires_defauts(categorie):
    """Liste les paires (image de défaut, masque) réelles du split test."""
    paires = []
    gt_dir = Path(DATA_DIR, categorie, "ground_truth")
    for dossier in sorted(gt_dir.iterdir()):
        if not dossier.is_dir():
            continue
        for masque in sorted(dossier.glob("*_mask.png")):
            idx = masque.name.replace("_mask.png", "")
            image = Path(DATA_DIR, categorie, "test", dossier.name, f"{idx}.png")
            if image.exists() and masque.exists():
                paires.append((image, masque))
    print(f"  Défauts réels {categorie} : {len(paires)} paires (image + masque)")
    return paires


def charger_image_rgb(chemin, taille):
    """Charge une image en RGB uint8 redimensionnée."""
    image = Image.open(chemin).convert("RGB").resize((taille, taille), Image.Resampling.BILINEAR)
    return np.array(image)


def charger_masque_binaire(chemin, taille):
    """Charge un masque binaire {0,1} redimensionné (nearest)."""
    masque = Image.open(chemin).convert("L").resize((taille, taille), Image.Resampling.NEAREST)
    return (np.array(masque) > 127).astype(np.uint8)


def calculer_alpha(masque, flou):
    """Transforme un masque binaire en alpha feathered (bords adoucis)."""
    k = flou if flou % 2 == 1 else flou + 1
    k = max(3, min(k, (min(masque.shape[:2]) // 2) * 2 + 1))
    alpha = cv2.GaussianBlur(masque.astype(np.float32), (k, k), 0)
    if alpha.max() > 0:
        alpha = alpha / alpha.max()
    return alpha


def extraire_patch(defaut, masque, marge):
    """Extrait le rectangle du défaut (RGB + masque) et sa position."""
    ys, xs = np.where(masque > 0)
    if len(ys) == 0:
        return None
    y0 = max(0, ys.min() - marge)
    y1 = min(masque.shape[0], ys.max() + 1 + marge)
    x0 = max(0, xs.min() - marge)
    x1 = min(masque.shape[1], xs.max() + 1 + marge)
    return defaut[y0:y1, x0:x1].copy(), masque[y0:y1, x0:x1].copy(), int(x0), int(y0)

## 3. Génération, grille d'exemples et LOG

In [3]:
def coller_defaut(fond, patch, alpha, x, y):
    """Colle un patch sur le fond avec alpha blending (gère les bords)."""
    H, W = fond.shape[:2]
    h, w = patch.shape[:2]
    x0, y0 = max(0, x), max(0, y)
    x1, y1 = min(W, x + w), min(H, y + h)
    if x1 <= x0 or y1 <= y0:
        return fond
    px0, py0 = x0 - x, y0 - y
    sp = patch[py0:py0 + (y1 - y0), px0:px0 + (x1 - x0)].astype(np.float32)
    sa = alpha[py0:py0 + (y1 - y0), px0:px0 + (x1 - x0)][..., None]
    roi = fond[y0:y1, x0:x1].astype(np.float32)
    fond[y0:y1, x0:x1] = np.clip(roi * (1 - sa) + sp * sa, 0, 255).astype(np.uint8)
    return fond


def ajouter_variation_globale(image, rng):
    """Applique une légère variation de luminosité et un bruit gaussien."""
    gain = 1.0 + rng.uniform(-VARIATION_LUMINOSITE, VARIATION_LUMINOSITE)
    bruit = rng.normal(0, BRUIT_GAUSS, image.shape)
    return np.clip(image.astype(np.float32) * gain + bruit, 0, 255).astype(np.uint8)


def generer_une_image(normaux, paires, rng):
    """Copie un défaut réel (via son masque) sur une image normale."""
    normal_path = normaux[int(rng.integers(0, len(normaux)))]
    fond = charger_image_rgb(normal_path, TAILLE)
    image_defaut, masque_path = paires[int(rng.integers(0, len(paires)))]
    defaut = charger_image_rgb(image_defaut, TAILLE)
    masque = charger_masque_binaire(masque_path, TAILLE)
    extrait = extraire_patch(defaut, masque, MARGE_BBOX)
    if extrait is None:
        return fond
    patch, mask_patch, x0, y0 = extrait
    alpha = calculer_alpha(mask_patch, FLOU_MASQUE) * FORCE_DEFAUT
    if rng.random() < 0.5:
        patch = patch[:, ::-1]
        alpha = alpha[:, ::-1]
    dx = int(rng.integers(-JITTER_MAX, JITTER_MAX + 1))
    dy = int(rng.integers(-JITTER_MAX, JITTER_MAX + 1))
    fond = coller_defaut(fond, patch, alpha, x0 + dx, y0 + dy)
    return ajouter_variation_globale(fond, rng)


def generer_categorie(categorie, rng):
    """Génère N images synthétiques pour une catégorie et retourne leurs chemins."""
    normaux = sorted(Path(DATA_DIR, categorie, "train", "good").glob("*.png"))
    paires = lister_paires_defauts(categorie)
    if not normaux or not paires:
        print(f"[!] Données manquantes pour {categorie}, catégorie ignorée")
        return []
    dossier = Path(SYNTH_DIR, categorie)
    dossier.mkdir(parents=True, exist_ok=True)
    for ancien in dossier.glob("synth_*.png"):
        ancien.unlink()
    chemins = []
    for i in range(N_SYNTH_PAR_CATEGORIE):
        image = generer_une_image(normaux, paires, rng)
        chemin = dossier / f"synth_{i:03d}.png"
        Image.fromarray(image).save(chemin)
        chemins.append(chemin)
        if (i + 1) % 20 == 0:
            print(f"  {categorie} : {i + 1}/{N_SYNTH_PAR_CATEGORIE}")
    print(f"  {categorie} : {len(chemins)} images sauvegardées dans {dossier}")
    return chemins


def sauvegarder_grille(categorie, chemins_synth):
    """Grille 3 lignes : normaux, défauts réels, défauts synthétiques."""
    n_col = N_EXEMPLES_GRILLE
    normaux = sorted(Path(DATA_DIR, categorie, "train", "good").glob("*.png"))[:n_col]
    reels = []
    for dossier in sorted(Path(DATA_DIR, categorie, "test").iterdir()):
        if dossier.is_dir() and dossier.name != "good":
            reels.extend(sorted(dossier.glob("*.png")))
            if len(reels) >= n_col:
                break
    reels = reels[:n_col]
    synth = list(chemins_synth)[:n_col]
    fig, axes = plt.subplots(3, n_col, figsize=(n_col * 2, 6))
    titres = ["Normaux", "Défauts réels", "Défauts synthétiques"]
    for i, (titre, liste) in enumerate(zip(titres, [normaux, reels, synth])):
        axes[i, 0].set_ylabel(titre)
        for j in range(n_col):
            axes[i, j].axis("off")
            if j < len(liste):
                axes[i, j].imshow(Image.open(liste[j]).convert("RGB"))
    fig.suptitle(f"Exemples synthétiques — {categorie}")
    fig.tight_layout()
    chemin = Path(FIGURES_DIR, f"fig_synthetiques_exemples_{categorie}.png")
    fig.savefig(chemin, dpi=120)
    plt.close(fig)
    print(f"  Grille sauvegardée : {chemin}")


def ecrire_log(compteurs):
    """Écrit un LOG avec le nombre d'images générées par catégorie."""
    total = sum(compteurs.values())
    lignes = [
        "=== LOG génération de défauts synthétiques IND-3 ===",
        f"Date : {datetime.now().isoformat()}",
        f"Seed : {SEED}",
        "Méthode : copy-paste de masques réels + alpha blending",
        f"Taille : {TAILLE}x{TAILLE}",
        f"Device détecté : {DEVICE}",
        "",
        "Images générées par catégorie :",
    ]
    lignes += [f"  {cat} : {n}" for cat, n in compteurs.items()]
    lignes += ["", f"TOTAL : {total}"]
    chemin = Path(SYNTH_DIR, "LOG.txt")
    chemin.write_text("\n".join(lignes) + "\n", encoding="utf-8")
    print(f"LOG écrit : {chemin}")

## 4. Exécution

In [4]:
compteurs = {}
chemins_par_categorie = {}

for categorie in CATEGORIES:
    print(f"\n=== Génération {categorie} ===")
    rng = np.random.default_rng(SEED + CATEGORIES.index(categorie))
    chemins = generer_categorie(categorie, rng)
    chemins_par_categorie[categorie] = chemins
    compteurs[categorie] = len(chemins)

print("\n=== Grilles d'exemples ===")
for categorie in CATEGORIES:
    if chemins_par_categorie[categorie]:
        sauvegarder_grille(categorie, chemins_par_categorie[categorie])

ecrire_log(compteurs)
print("\nTerminé.")


=== Génération bottle ===
  Défauts réels bottle : 63 paires (image + masque)
  bottle : 20/120
  bottle : 40/120
  bottle : 60/120
  bottle : 80/120
  bottle : 100/120
  bottle : 120/120
  bottle : 120 images sauvegardées dans synthetiques/bottle

=== Génération cable ===
  Défauts réels cable : 92 paires (image + masque)
  cable : 20/120
  cable : 40/120
  cable : 60/120
  cable : 80/120
  cable : 100/120
  cable : 120/120
  cable : 120 images sauvegardées dans synthetiques/cable

=== Génération transistor ===
  Défauts réels transistor : 40 paires (image + masque)
  transistor : 20/120
  transistor : 40/120
  transistor : 60/120
  transistor : 80/120
  transistor : 100/120
  transistor : 120/120
  transistor : 120 images sauvegardées dans synthetiques/transistor

=== Grilles d'exemples ===
  Grille sauvegardée : figures/fig_synthetiques_exemples_bottle.png
  Grille sauvegardée : figures/fig_synthetiques_exemples_cable.png
  Grille sauvegardée : figures/fig_synthetiques_exemples_tra